In [3]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster, GroupedLayerControl ####, FeatureGroup
from folium import FeatureGroup
# Your existing pilot input code...

# Load airport data
df = pd.read_csv("https://davidmegginson.github.io/ourairports-data/airports.csv")
df = df[df["iso_country"] == "CA"].copy()  # Only Canadian aerodromes
df.dropna(subset=["ident", "latitude_deg", "longitude_deg"], inplace=True)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude_deg, df.latitude_deg), crs="EPSG:4326")

# Define airport icon styles
ICON_STYLE = {
    "heliport": {"icon": "helicopter", "color": "green"},
    "seaplane_base": {"icon": "ship", "color": "cadetblue"},
    "small_airport": {"icon": "circle", "color": "gray"},
    "medium_airport": {"icon": "plane", "color": "blue"},
    "large_airport": {"icon": "plane", "color": "darkblue"},
    "default": {"icon": "map-marker", "color": "lightgray"},
}

# Create map
m = folium.Map(location=[44.0, -79.0], zoom_start=8)

# Create feature groups for each province
province_parents = {}
province_names = sorted(gdf["iso_region"].unique())

# Create the feature groups
for prov in province_names:
    fg = FeatureGroup(name=prov).add_to(m)
    province_parents[prov] = fg

    # Group by airport type within the province
    for atype, sub in gdf[gdf['iso_region'] == prov].groupby("type"):
        cluster = MarkerCluster().add_to(fg)
        style = ICON_STYLE.get(atype, ICON_STYLE["default"])
        
        # Add markers to the cluster
        for _, row in sub.iterrows():
            lat, lon = row.geometry.y, row.geometry.x
            folium.Marker(
                [lat, lon],
                tooltip=f"{row['ident']} • {row['name']}",
                popup=f"{row['ident']} – {row['name']}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}",
                icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="fa")
            ).add_to(cluster)

# Add GroupedLayerControl
GroupedLayerControl(
    layers=list(province_parents.values()),
    grouped=True,
    collapsed=False
).add_to(m)

# Save or display map
m.save('canadian_aerodromes_map.html')
m

TypeError: GroupedLayerControl.__init__() missing 1 required positional argument: 'groups'